# 02 — Alex's Morning Scan

**Notebook 2 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisite: [`01-foundations-techtrade-and-analysis.ipynb`](./01-foundations-techtrade-and-analysis.ipynb) — you should already have a working `.venv_win`, a configured `fmp_cached_api_key`, and have seen `obb.techtrade.about()` return cleanly.

---

## Recap: who's Alex?

Alex is the software engineer from notebook 01: literate in Python, day-trades on the side, has been losing more than he likes by trading on hunches, and now wants a system. Notebook 01 showed him the breadth of `obb.techtrade.*`. This notebook is the **first deep workflow**: his morning routine.

## What this notebook covers

By the end, Alex has produced his **morning trade-plan workbook** — a 6-sheet Excel file containing the day's top setups, ranked by conviction, with the audit trail (which indicators voted long/short on each ticker, what entry/stop/target prices, what position size at 1% account risk). This is the artifact he can stare at over coffee before the market opens.

**Mapped to the README's command surface (§ Commands):**

| Step | Command | What it gives Alex |
|---|---|---|
| 1 | `obb.techtrade.segments()` | The 11 GICS sectors he could scan today |
| 2 | `obb.techtrade.movers(segment=...)` | The top movers within each sector (sanity check) |
| 3 | `obb.techtrade.scan(metric=..., top_n=..., preset=..., risk=...)` | **The headline call** — ranked actionable plans across all 11 sectors |
| 4 | `obb.techtrade.export(plans=...)` | The printable Excel workbook |

**Wall-clock budget:** ~3-8 minutes if your `fmp_cached` cache is warm; up to ~20 minutes cold (Excel export is the slow step). All cells call real `fmp_cached`; nothing is faked.

## What this notebook is NOT

- **Not a backtest.** Robustness gating lives in notebook 04 (`obb.techtrade.validate`).
- **Not a single-position deep dive.** Notebook 03 takes one ticker from this scan and shows the full plan → orders → paper-fill chain.
- **Not investment advice.** Every plan this notebook produces is *research / paper-trading material*. Alex pulls the trigger; the engine never does.

## 1. Setup probe

First a 5-second sanity check that the environment is the same one notebook 01 set up. If anything fails, fix it per `01-foundations` §1.

In [ ]:
import importlib.util
import sys
from pathlib import Path

REQUIRED = ["openbb", "openbb_techtrade", "openbb_fmp_cached", "openpyxl", "pandas_ta_classic"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"Missing required packages: {missing}. See 01-foundations §1.")

settings = Path.home() / ".openbb_platform" / "user_settings.json"
if not settings.exists():
    raise RuntimeError(f"Missing credentials file: {settings}")

print(f"Python {sys.version.split()[0]} | {len(REQUIRED)} required packages present | credentials configured")

In [ ]:
# First obb import is ~5-10s (openbb.build()); later cells are instant.
from openbb import obb
from datetime import date

print(f"obb loaded; today is {date.today().isoformat()}")
print(f"techtrade version: {obb.techtrade.about().results.get('extension_version', '?')}")

## 2. The sector universe

`obb.techtrade.segments()` returns the 11 GICS sectors. Each sector knows its benchmark ETF (XLK for IT, XLF for Financials, etc.) and how its universe is resolved — by default, the ETF's current holdings. This is the input space for the scan: today's setups will be sourced from these sectors.

In [ ]:
segments = obb.techtrade.segments().results
print(f"Sectors available: {len(segments)}\n")
for s in segments:
    print(f"  {s.segment:<25} benchmark={s.benchmark_etf or '(none)':<6} source={s.universe_source} top_n={s.top_n}")

### Reading the output

Every sector has a `benchmark_etf` (the Select Sector SPDR — XLK, XLF, XLV, ...) and a `universe_source` (`etf_holdings` by default). When a `scan` runs, it expands each ETF into its current holdings and ranks them. Today's universe is therefore *not hardcoded* — it floats with the ETF's quarterly rebalances.

## 3. Sanity-check one sector

Before scanning all 11, Alex pokes one sector to see what the mover ranking looks like. `metric="pct_change"` ranks by today's price move; `top_n=10` keeps the top 10 movers within the sector.

In [ ]:
# ~3-10s on a warm cache; longer cold.
movers_it = obb.techtrade.movers(
    segment="Information Technology",
    metric="pct_change",
    top_n=10,
).results

# `movers_it` is a list[MoverList]; one entry for the segment, containing the ranked tickers.
ml = movers_it[0]
print(f"Segment: {ml.segment}  |  as_of: {ml.as_of}  |  movers: {len(ml.movers)}\n")
for m in ml.movers:
    print(f"  #{m.rank:<2} {m.symbol:<6} pct_change={m.pct_change:+.2f}%  volume={m.volume:,.0f}")

### What Alex looks for here

- **A reasonable spread of `pct_change`** — if every name is at ±0.1% the market is flat and today probably isn't a setup day.
- **Volume should not be zero** — a top-mover with no volume is illiquid and Alex can't safely take a position there even if the signal triggers.
- **No surprise names** — he should recognize most of the top-10 in IT (Apple / Microsoft / Nvidia / etc.). If a microcap he's never heard of shows up at #1, that's a flag for further inspection.

## 4. The morning scan

Now the headline command. `obb.techtrade.scan(...)` runs the full pipeline (movers → indicator panel → confluence signal → rule → sizing → orders → recommendation) across **all 11 sectors** and ranks the resulting plans by conviction.

**Argument choices below — these are Alex's morning defaults:**

| Arg | Value | Why |
|---|---|---|
| `metric="pct_change"` | Rank movers within each sector by today's % move | Standard "what's moving" filter |
| `top_n=5` | Keep top 5 movers per sector (= up to 55 candidates) | Bounds wall-clock; 5 per sector is plenty of diversity |
| `preset="trend_follow"` | Confluence weights tilted to trend signals | Default; tomorrow's notebook 03 will demo `mean_revert` and `breakout` |
| `risk=0.01` | 1% of notional risked per trade | Conventional position-sizing; § 7 below shows how to change this |

In [ ]:
# Wall-clock: 1-3 minutes warm; 5-15 minutes cold. This is the morning's heaviest call.
scan_result = obb.techtrade.scan(
    metric="pct_change",
    top_n=5,
    preset="trend_follow",
    risk=0.01,
)
plans = scan_result.results
print(f"scan returned {len(plans)} plans across {len({p.segment for p in plans})} sectors.")

### If you got 0 plans

Not a bug. It means every candidate today scored below the entry threshold (default `|score| >= 0.4`). Real markets have flat days. Try:

- A different `preset` — `mean_revert` will flag pullbacks the trend-follower ignored.
- A lower entry threshold via the `plan` command directly (advanced — see notebook 03).
- Wait a day. "No setup" is a legitimate outcome of discipline.

## 5. Quick-look ranking

Before the Excel export, Alex glances at the top of the list. Conviction is bucketed as `High`/`Medium`/`Low` from the absolute confluence score (the README § Commands describes the bucketing).

In [ ]:
import pandas as pd

rows = [
    {
        "symbol": p.symbol,
        "segment": p.segment[:20],
        "score": p.signal.score,
        "action": p.recommendation.action,
        "conviction": p.recommendation.conviction,
        "entry": float(p.recommendation.entry_price),
        "stop": float(p.recommendation.stop_price),
        "target": float(p.recommendation.target_price),
        "r:r": p.recommendation.risk_reward,
        "qty": float(p.recommendation.position_size),
    }
    for p in plans
]

df = pd.DataFrame(rows)
if df.empty:
    print("No plans today; nothing to rank.")
else:
    # Show the top 15 by absolute score.
    df_sorted = df.reindex(df["score"].abs().sort_values(ascending=False).index)
    df_sorted.head(15).reset_index(drop=True)

### Reading the table

- `score` is in `[-1, +1]`. Sign = direction (positive = long, negative = short). Magnitude = confluence strength.
- `action` is the bucketed call: `BUY` / `SELL_SHORT` / `HOLD/FLAT`.
- `conviction` thresholds (from PRD §12.2): `High` if `|score| >= 0.7`, `Medium` if `|score| >= 0.4`, else `Low` (filtered out by the entry threshold).
- `r:r` is the reward-to-risk ratio at the stop/target levels. Below 2.0 means the trade has to be right more than half the time to break even on expectation — a red flag even with a high conviction score.
- `qty` is the position size at the 1% risk Alex specified. It scales inversely with the stop distance: tight stop = bigger position; wide stop = smaller position.

## 6. Filter to today's actionable list

Alex doesn't take every signal. He filters to:

1. **`High` conviction only** (drops the medium-noise tier).
2. **`r:r >= 2.0`** (drops trades with bad reward-to-risk).
3. **`segment` diversity** — cap at 2 plans per sector so he isn't all-in on one sector's idiosyncrasies.

These three filters are his standing rules; they're not techtrade defaults. Edit this cell to match your own discipline.

In [ ]:
from collections import defaultdict

actionable: list = []
per_sector = defaultdict(int)
PER_SECTOR_CAP = 2

# Sort by absolute score descending so the strongest in each sector wins the cap.
for p in sorted(plans, key=lambda p: -abs(p.signal.score)):
    rec = p.recommendation
    if rec.conviction != "High":
        continue
    if rec.risk_reward < 2.0:
        continue
    if per_sector[p.segment] >= PER_SECTOR_CAP:
        continue
    actionable.append(p)
    per_sector[p.segment] += 1

print(f"Actionable after filters: {len(actionable)} / {len(plans)} plans\n")
for p in actionable:
    rec = p.recommendation
    print(f"  {p.symbol:<6} {p.segment[:18]:<18} score={p.signal.score:+.3f} {rec.action:<10} r:r={rec.risk_reward:.1f}  qty={rec.position_size}")

## 7. Why does each setup look good? The audit trail

techtrade's competitive advantage over a black-box scanner is that **every recommendation comes with its full vote attribution**. The `signal.votes` field shows which indicator in which family voted which direction with which weight. Alex can always answer "why long?"

In [ ]:
if not actionable:
    print("No actionable plans today; nothing to audit.")
else:
    p = actionable[0]
    print(f"--- Audit for {p.symbol} ({p.segment}) ---")
    print(f"Direction: {p.signal.direction}  |  Composite score: {p.signal.score:+.3f}\n")
    print(f"{'family':<12} {'name':<14} {'vote':>6} {'weight':>8} {'contrib':>10}")
    for v in sorted(p.signal.votes, key=lambda v: -abs(v.vote * v.weight)):
        contrib = v.vote * v.weight
        print(f"{v.family:<12} {v.name:<14} {v.vote:+.2f}   {v.weight:.2f}     {contrib:+.4f}")
    print(f"\nReasoning:  {p.recommendation.reasoning}")
    print(f"Caveats:    {p.recommendation.caveats}")

### How to read the audit

- `vote` is the indicator's directional reading on `[-1, +1]`.
- `weight` is what the preset assigns to that family (trend 0.40 / momentum 0.25 / volatility 0.20 / volume 0.15 by default).
- `contrib` is `vote * weight` — the indicator's signed contribution to the composite score.
- The **`reasoning`** field is a deterministic natural-language summary built from the top contributors. No LLM — if the same panel + weights are fed again, the same reasoning text comes out.
- The **`caveats`** field flags risk concerns the engine noticed (low volume, wide stop, opposing high-weight indicator, etc.).

## 8. Export to Excel (the morning artifact)

Alex's final morning step is the 6-sheet Excel workbook. Each sheet is the same data sliced differently:

| Sheet | What's on it |
|---|---|
| **Recommendations** | One row per plan with the action/conviction/levels (with the "research/paper — not investment advice" disclaimer banner) |
| **Levels** | Entry / stop / target / stop distance % / target distance % / ATR per plan |
| **Reasoning** | The full `reasoning` text + `top_factors` per plan (the audit trail he saw above) |
| **Orders** | The broker-ready order legs (entry + stop + target + time-exit) per plan |
| **Fills** | Paper fills if `scan(simulate=True)` returned them (the v1 router does not yet — see notebook 03) |
| **Summary** | One-row dashboard: total plans, score distribution, sector breakdown |

Default output path is `Analysis/exports/techtrade_<date>.xlsx`. Conditional formatting on the Recommendations sheet color-codes action and risk-reward.

In [ ]:
# Export ONLY the actionable plans, not every signal that came back.
# Wall-clock: 5-30s depending on plan count + I/O.
export_result = obb.techtrade.export(
    plans=actionable,
    path=None,  # use the default `Analysis/exports/techtrade_<date>.xlsx`
    engine="openpyxl",  # ships with the bare install; `xlsxwriter` is the optional alternative
)
xlsx_path = Path(export_result.results)
print(f"Workbook written to: {xlsx_path}")
print(f"Size: {xlsx_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Peek at the sheet names + the first few rows of Recommendations so Alex sees
# what landed without opening Excel.
import openpyxl

wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
print(f"Sheets ({len(wb.sheetnames)}): {wb.sheetnames}\n")

rec = wb["Recommendations"]
print("--- Recommendations sheet (first 4 rows) ---")
for i, row in enumerate(rec.iter_rows(values_only=True, max_row=4)):
    cells = [str(c)[:18] if c is not None else "" for c in row[:8]]
    print("  " + " | ".join(cells))
wb.close()

## 9. Variation: try the other two presets

The same scan with a different preset gives a different list. `mean_revert` flags pullbacks; `breakout` flags volatility-led setups. Alex sometimes runs all three and looks at the **overlap** — a name that shows up under multiple presets is a stronger signal than a name picked by just one.

**Heads-up:** each preset takes another full scan cycle. Two more presets = another 2-6 minutes warm-cache.

In [ ]:
# Re-scan with mean_revert and breakout; compare the High-conviction symbols.
preset_picks = {"trend_follow": {p.symbol for p in actionable}}

for preset in ("mean_revert", "breakout"):
    res = obb.techtrade.scan(metric="pct_change", top_n=5, preset=preset, risk=0.01).results
    high = {p.symbol for p in res if p.recommendation.conviction == "High"}
    preset_picks[preset] = high
    print(f"{preset:<14} High-conviction picks: {sorted(high)}")

# Overlap analysis.
all_picks = set().union(*preset_picks.values())
for sym in sorted(all_picks):
    by_preset = [name for name, s in preset_picks.items() if sym in s]
    print(f"  {sym:<6} flagged by {len(by_preset)}/3 presets: {by_preset}")

### When to trust the overlap

- **3/3 overlap**: rare, strong cross-preset agreement. The name is moving in a way that satisfies trend-followers AND mean-reverters AND breakout traders all at once. Worth a deeper look in notebook 03.
- **2/3 overlap**: common; usually means a trending name that's also pulling back (trend + mean-revert agree) or breaking out (trend + breakout agree). Decent signal.
- **1/3 overlap**: the preset matters. Treat the pick as preset-specific — a `mean_revert`-only flag means Alex needs to be in a fade-the-rally mood that day.

## 10. Variation: change the risk knob

`risk=0.01` is conventional. `risk=0.005` is conservative (half the position size, smaller losses on a stopped trade); `risk=0.02` is aggressive (twice the size, twice the loss). The `qty` column in the table scales linearly with risk.

Alex's rule of thumb: **never run the morning scan with a `risk` value he hasn't thought through.** It's the single fastest way to blow up an account.

In [ ]:
# Show how qty scales for the top actionable plan at three risk levels.
# Re-running scan() at each risk is expensive; instead, we observe that the engine
# computes qty = round(risk * notional / (entry - stop)). Linear scaling = quick demo.
if actionable:
    p = actionable[0]
    rec = p.recommendation
    base_risk = 0.01
    base_qty = float(rec.position_size)
    print(f"{p.symbol} at default risk {base_risk:.1%}: qty = {base_qty}")
    for r in (0.005, 0.015, 0.02):
        scaled = base_qty * (r / base_risk)
        print(f"  same plan at risk {r:.1%}: qty = {scaled:.0f}")

## 11. Wrap-up checklist

Before the bell rings, Alex's discipline says verify these five things from the morning scan:

- [ ] **The workbook opened cleanly in Excel** — no missing sheets, the disclaimer is visible on the Recommendations sheet.
- [ ] **At least one filter narrowed the list.** If `len(actionable) == len(plans)` then his filters are no-ops; he should tighten them before relying on the output.
- [ ] **Reasoning text is non-empty** for every actionable plan. A blank `reasoning` means the audit trail didn't produce a narrative; treat that plan as suspect.
- [ ] **No top-3 symbol comes from a sector he wouldn't trade today.** (Energy on FOMC day, biotech on FDA-meeting day, etc.)
- [ ] **R:R passes a gut check.** A `r:r = 5.0` looks great but probably means an unrealistically far target.

If any of those fail, **re-scan with different args** or **skip the day**. "Don't trade" is a valid output of a disciplined morning routine.

---

## What's next

- **Notebook 03 — Single Position Deep Dive**: take ONE ticker from today's actionable list, compute its individual `plan`, materialize the `orders` legs, and paper-fill them forward with `simulate`. This is where Alex sees the no-look-ahead fill discipline up close.
- **Notebook 04 — The Validation Gate**: take ONE plan and run `obb.techtrade.validate(plan, method="wfo")`. Walk-forward folds + PBO + DSR + a verdict. The anti-overfit gate.
- **Notebook 05 — Per-Sector Tuning**: `obb.techtrade.tune(segment)` with the `[tuneta]` extra. The optional path that proposes better indicator periods per sector.
- **Notebook 06 — Audit and Replay**: load yesterday's xlsx, replay what Alex actually did vs what the engine suggested, write a journal entry.

---

*End of notebook 02. Series: "A Developer Guide to Disciplined Trading". Maintained on the `trading_technicals` branch of `prajoria/OpenBB`.*